# 01 — Data Preprocessing
Carica le feature grezze, applica PCA sul visual, salva i dati pronti per i modelli.


In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from collections import Counter

# ── PATHS ────────────────────────────────────────────────────────────
PROJECT_PATH  = 'Thesis_Data'
LABELS_PATH   = 'videos_with_sentiment_labels.csv'
OUTPUT_PATH   = 'processed_data'          # cartella dove salvare i .npy

import os
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [2]:
# ── CARICA DATI GREZZI ───────────────────────────────────────────────
df           = pd.read_csv(LABELS_PATH)
visual_dict  = np.load(f'{PROJECT_PATH}/visual_features_clip.npy',  allow_pickle=True).item()
audio_dict   = np.load(f'{PROJECT_PATH}/audio_features_vggish.npy', allow_pickle=True).item()

X_visual, X_audio, y_labels = [], [], []

for _, row in df.iterrows():
    v_id  = row['video_id']
    label = row['majority_sentiment']
    if v_id in visual_dict and v_id in audio_dict:
        X_visual.append(visual_dict[v_id])
        X_audio.append(audio_dict[v_id])
        y_labels.append(label)

X_visual = np.array(X_visual)   # (N, 2048)
X_audio  = np.array(X_audio)    # (N, 128)
y_labels = np.array(y_labels)

le       = LabelEncoder()
y_encoded = le.fit_transform(y_labels)

print(f'Campioni totali : {len(y_encoded)}')
print(f'Feature visive  : {X_visual.shape}')
print(f'Feature audio   : {X_audio.shape}')
print(f'Classi          : {le.classes_}')

Campioni totali : 446
Feature visive  : (446, 2048)
Feature audio   : (446, 128)
Classi          : ['Negative' 'Neutral' 'Positive']


In [3]:
# ── DISTRIBUZIONE CLASSI ─────────────────────────────────────────────
dist = Counter(y_encoded)
for cls, count in sorted(dist.items()):
    print(f'  {le.classes_[cls]:10s}: {count:3d} campioni ({count/len(y_encoded)*100:.1f}%)')

  Negative  :  52 campioni (11.7%)
  Neutral   : 136 campioni (30.5%)
  Positive  : 258 campioni (57.8%)


In [4]:
# ── PCA SUL VISUAL (2048 → 64) ───────────────────────────────────────
# Nota: la PCA viene fittata sull'intero dataset QUI.
# Dentro ogni fold i modelli useranno queste feature già ridotte.
# L'audio VGGish (128-dim) è già compatto: nessuna PCA necessaria.

PCA_VISUAL_COMPONENTS = 64

pca_v = PCA(n_components=PCA_VISUAL_COMPONENTS, random_state=42)
X_visual_pca = pca_v.fit_transform(X_visual)

explained = pca_v.explained_variance_ratio_.sum()
print(f'PCA visual: {X_visual.shape[1]} → {PCA_VISUAL_COMPONENTS} componenti')
print(f'Varianza spiegata: {explained*100:.1f}%')

# L'audio rimane invariato
X_audio_final = X_audio.copy()
print(f'Audio invariato  : {X_audio_final.shape}')

PCA visual: 2048 → 64 componenti
Varianza spiegata: 71.7%
Audio invariato  : (446, 128)


In [5]:
# ── SALVA I DATI PROCESSATI ──────────────────────────────────────────
np.save(f'{OUTPUT_PATH}/X_visual_pca.npy',  X_visual_pca)   # (N, 64)
np.save(f'{OUTPUT_PATH}/X_audio.npy',       X_audio_final)  # (N, 128)
np.save(f'{OUTPUT_PATH}/y_encoded.npy',     y_encoded)       # (N,)
np.save(f'{OUTPUT_PATH}/label_classes.npy', le.classes_)     # ['Negative','Neutral','Positive']

print('✅ Dati salvati in:', OUTPUT_PATH)
print(f'   X_visual_pca  : {X_visual_pca.shape}')
print(f'   X_audio       : {X_audio_final.shape}')
print(f'   y_encoded     : {y_encoded.shape}')
print(f'   label_classes : {le.classes_}')

✅ Dati salvati in: processed_data
   X_visual_pca  : (446, 64)
   X_audio       : (446, 128)
   y_encoded     : (446,)
   label_classes : ['Negative' 'Neutral' 'Positive']
